# Simulation Analysis

## Imports

In [ ]:
# Important modules
import os
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import Normalize
from IPython.display import Image

# Kinetic misalignment functions
from kin_mis_utils import evolution_kinetic_mis, genspec_kinetic_mis, find_all_axitons

# Run simulations
from pyaxions import simgen as sg

# Deal with the measurement files from the simulations
from pyaxions import jaxions as pa
from pyaxions import spaxcreate as sp
from pyaxions import spectrum as spec
import importlib
importlib.reload(pa)

# Some plot settings
import matplotlib as mpl
mpl.rcParams['mathtext.rm'] = 'serif'
mpl.rcParams['mathtext.fontset'] = 'cm'
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.size'] = 16
mpl.rcParams['text.usetex'] = True

## General 

In [ ]:
# Simulation parameters
N = 128 # Points/dimension
L = 3 # Box length in L₁ units

# Axion parameters
fAGeV = 1e+11 # Axion decay constant in GeV
theta1 = 5.40 # Initial field value of zero mode
vheta1 = 102.70 # Initial velocity value of zero mode

In [ ]:
# Load measurement file
path = os.getcwd()

file_path = f"/out_N{N}_L{L}_fA{fAGeV:.0e}_theta{theta1:.2f}_vheta{vheta1:.2f}"
file_path = "/out"

mf = pa.findmfiles(path +  file_path)
mask = pa.gml(mf, 'psp?')

In [ ]:
# Measurements
R = pa.gml(mf, 'R') # Scale factor in R₁
tau = pa.gml(mf, 'ct') # Conformal time 
T = pa.gml(mf, 'T') # Temperature in MeV
theta_maps = pa.gml(mf, 'maptheta') # Field 
theta_proj_maps = pa.gml(mf, '2DmapP') # Field projection
vheta_maps = (pa.gml(mf, 'mapvheta')-theta_maps)/R[:, None, None] # Velocity 
theta_Edens_maps = pa.gml(mf, 'mapEdens') # Energy density 
mA = pa.gml(mf, 'massA') # Axion mass, conformal mass: ma*r
eA = pa.gml(mf, 'eA') # Axion energy
sizeN = pa.gm(mf[0], 'N') # Points/dimension
sizeL = pa.gm(mf[0], 'L') # Box length in L₁ units
nmodes = pa.phasespacedensityBOX(sizeN) # Phase space modes (averages over # of modes in each shell)
esps = pa.gml(mf[mask], 'espV_0')/(nmodes*(R[mask, None]**2)) # Axion potential energy density spectrum divided by frequency
psps = pa.gml(mf[mask], 'psp')/(eA[mask, None]**2 * nmodes) # Axion energy density contrast spectrum
pspVs = pa.gml(mf[mask], 'pspmaskedV')/(eA[mask, None]**2 * nmodes) # Vil masked axion energy density contrast spectrum
psp4s = pa.gml(mf[mask], 'pspmasked_4.00')/(eA[mask, None]**2 * nmodes) # Axit masked axion energy density contrast spectrum
s = spec.espevol(mf) # Spectral values
k_values = s.avek # Momentum values

In [ ]:
tau_idx = np.argmin((tau-4)**2) # Choose interesting time step

In [ ]:
def mod_func(x):
    """
    Shift input by π to center the range around 0. Then apply modulo 2π to wrap it within [0, 2π).
    Finally, subtract π to shift the range back to (-π, π].

    Args:
        x (float): Angle in radians.

    Returns:
        (float): Angle wrapped into (-π, π].
    
    """
    
    return (x + np.pi) % (2 * np.pi) - np.pi

In [ ]:
# Plot field distribution
plt.figure(figsize = (6, 6))
plt.imshow(mod_func(theta_maps[tau_idx]), cmap = pa.thetacmap, vmin = -np.pi, vmax = np.pi)
plt.colorbar(label = r'$\theta$')
plt.xlabel(r'$x$')  
plt.ylabel(r'$y$')  
plt.title(r'Field Distribution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
# Get current axis limits to place text in data units
xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()
plt.text(
    xmin + 0.05 * (xmax - xmin), ymax - 0.05 * (ymax - ymin),
    fr'$\tau = {tau[tau_idx]:.2f}$', 
    color = 'white', fontsize = 12,
    verticalalignment = 'top',
    bbox = dict(facecolor = 'black', alpha = 0.6, boxstyle = 'round')
)
plt.show()
plt.close()

In [ ]:
# Plot field velocity distribution
plt.figure(figsize = (6, 6))
plt.imshow(vheta_maps[tau_idx], cmap = pa.thetacmap)
plt.colorbar(label = r'$\theta^\prime$')
plt.xlabel(r'$x$')  
plt.ylabel(r'$y$')  
plt.title(r'Field Velocity Distribution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
# Get current axis limits to place text in data units
xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()
plt.text(
    xmin + 0.05 * (xmax - xmin), ymax - 0.05 * (ymax - ymin),
    fr'$\tau = {tau[tau_idx]:.2f}$', 
    color = 'white', fontsize = 12,
    verticalalignment = 'top',
    bbox = dict(facecolor = 'black', alpha = 0.6, boxstyle = 'round')
)
plt.show()
plt.close()

In [ ]:
# Zero mode (average)
theta_avg = []  
vheta_avg = []

for i in np.arange(len(theta_maps)):
    theta_avg.append(np.mean(theta_maps[i]))
    vheta_avg.append(np.mean(vheta_maps[i]))
    
theta_avg = np.asarray(theta_avg)
vheta_avg = np.asarray(vheta_avg)

In [ ]:
# Fluctuation field
fluc_maps = theta_maps-theta_avg[:, np.newaxis, np.newaxis]

In [ ]:
# Plot field fluctuation distribution
plt.figure(figsize = (6, 6))
plt.imshow(fluc_maps[tau_idx], cmap = pa.thetacmap)
plt.colorbar(label = r'$\Delta\theta$')
plt.xlabel(r'$x$')  
plt.ylabel(r'$y$')  
plt.title(r'Field Fluctuation Distribution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
# Get current axis limits to place text in data units
xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()
plt.text(
    xmin + 0.05 * (xmax - xmin), ymax - 0.05 * (ymax - ymin),
    fr'$\tau = {tau[tau_idx]:.2f}$', 
    color = 'white', fontsize = 12,
    verticalalignment = 'top',
    bbox = dict(facecolor = 'black', alpha = 0.6, boxstyle = 'round')
)
plt.show()
plt.close()

In [ ]:
# Plot energy density distribution
plt.figure(figsize = (6, 6))
plt.imshow(np.log(theta_Edens_maps[tau_idx]), cmap = pa.thetacmap)
plt.colorbar(label = r'$\rho$')
plt.xlabel(r'$x$')  
plt.ylabel(r'$y$')  
plt.title(r'Energy Density Distribution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
# Get current axis limits to place text in data units
xmin, xmax = plt.xlim()
ymin, ymax = plt.ylim()
plt.text(
    xmin + 0.05 * (xmax - xmin), ymax - 0.05 * (ymax - ymin),
    fr'$\tau = {tau[tau_idx]:.2f}$', 
    color = 'white', fontsize = 12,
    verticalalignment = 'top',
    bbox = dict(facecolor = 'black', alpha = 0.6, boxstyle = 'round')
)
plt.show()
plt.close()

In [ ]:
# Plot number density over time
plt.figure(figsize = (6, 6))
plt.plot(tau, eA/mA * R**3, color = 'k') # Constant comoving number density 
plt.yscale('log')
plt.xlabel(r'$\tau$') 
plt.ylabel(r'$n_a$')  
plt.title(r'Number Density Evolution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
plt.show()
plt.close()

In [ ]:
# Random grid point
tau_rand, R_rand, _, theta_rand, vheta_rand, _ = np.loadtxt(path + file_path + '/sample.txt', unpack = True)

fig, ax = plt.subplots(1, 2, figsize = (12, 6))
ax[0].plot(tau_rand, (theta_rand/R_rand))
ax[0].set_xlabel(r'$\tau$')
ax[0].set_ylabel(r'$\theta$')
ax[1].plot(tau_rand, (vheta_rand-theta_rand/R_rand)/R_rand)
ax[1].set_xlabel(r'$\tau$')
ax[1].set_ylabel(r'$\theta^\prime$')
fig.suptitle(r'Evolution of Random Grid Point' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
plt.tight_layout()
plt.show()
plt.close()

## Animations

### Create

In [ ]:
create_ani = 0

In [ ]:
if create_ani:
    # Create animation of evolution of field distribution
    total_frames = len(theta_maps) # Shorten if necessary
    num_frames = 100
    indices = np.linspace(0, total_frames - 1, num_frames, dtype = int)
    
    # Plot
    fig, ax = plt.subplots(figsize = (6, 6))
    norm = Normalize(vmin = -np.pi, vmax = np.pi)
    im = ax.imshow(mod_func(theta_maps[0]), cmap = pa.thetacmap, norm = norm)
    cbar = plt.colorbar(im, ax = ax, shrink = 0.85)
    cbar.set_label(r'$\theta$')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_title(r'Field Distribution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
    fig.tight_layout(pad = 0.4)
    text = ax.text(0.05, 0.95, '', transform = ax.transAxes, color = 'white',
                   fontsize = 10, verticalalignment = 'top', bbox = dict(facecolor = 'black', alpha = 0.5, boxstyle = 'round'))
    
    def update(frame):
        i = indices[frame]
        im.set_data(mod_func(theta_maps[i]))
        text.set_text(fr"$\tau = {tau[i]:.2f}$")
        return [im, text]
    
    ani = FuncAnimation(fig, update, frames = num_frames, blit = True, interval = 100)
    writer = PillowWriter(fps = 3)
    
    ani.save(path +  file_path + "/animation.gif", writer = writer, dpi = 120)
    plt.close(fig)

In [ ]:
if create_ani:
    # Compute global min/max over selected frames
    global_min = np.min([np.min(fluc_maps[i]) for i in indices])
    global_max = np.max([np.max(fluc_maps[i]) for i in indices])
    
    # Create animation of evolution of field fluctuation distribution
    total_frames = len(fluc_maps) # Shorten if necessary
    num_frames = 100
    indices = np.linspace(0, total_frames - 1, num_frames, dtype = int)
    
    # Plot
    fig, ax = plt.subplots(figsize = (6, 6))
    im = ax.imshow(fluc_maps[0], cmap = pa.thetacmap, vmin = global_min, vmax = global_max)
    cbar = plt.colorbar(im, ax = ax, shrink = 0.85)
    cbar.set_label(r'$\Delta\theta$')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_title(
        r'Fluctuation Distribution' + '\n' + 
        r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
        fontsize = 12,
        fontweight = 'bold'
    )
    fig.tight_layout(pad = 0.4)
    text = ax.text(0.05, 0.95, '', transform = ax.transAxes, color = 'white',
                   fontsize = 10, verticalalignment = 'top', bbox = dict(facecolor = 'black', alpha = 0.5, boxstyle = 'round'))
    
    def update(frame):
        i = indices[frame]
        im.set_data(fluc_maps[i])
        text.set_text(fr"$\tau = {tau[i]:.2f}$")
        return [im, text]
    
    ani = FuncAnimation(fig, update, frames = num_frames, blit = True, interval = 100)
    writer = PillowWriter(fps = 3)
    
    ani.save(path +  file_path + "/fluc_animation.gif", writer = writer, dpi = 120)
    plt.close(fig)

In [ ]:
if create_ani:
    # Compute global min/max over selected frames
    global_min = np.min([np.min(theta_proj_maps[i]) for i in indices])
    global_max = np.max([np.max(theta_proj_maps[i]) for i in indices])
    
    # Create animation of evolution of energy density projection distribution
    total_frames = len(theta_maps) # Shorten if necessary
    num_frames = 100
    indices = np.linspace(0, total_frames - 1, num_frames, dtype = int)
    
    # Plot
    fig, ax = plt.subplots(figsize = (6, 6))
    im = ax.imshow(theta_proj_maps[0], cmap = pa.thetacmap, vmin = global_min, vmax = global_max)
    cbar = plt.colorbar(im, ax = ax, shrink = 0.85)
    cbar.set_label(r'$\sum_z \rho$')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$y$')
    ax.set_title(r'Energy Density Projection Distribution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
    fig.tight_layout(pad = 0.4)
    text = ax.text(0.05, 0.95, '', transform = ax.transAxes, color = 'white',
                   fontsize = 10, verticalalignment = 'top', bbox = dict(facecolor = 'black', alpha = 0.5, boxstyle = 'round'))
    
    def update(frame):
        i = indices[frame]
        im.set_data(theta_proj_maps[i])
        text.set_text(fr"$\tau = {tau[i]:.2f}$")
        return [im, text]
    
    ani = FuncAnimation(fig, update, frames = num_frames, blit = True, interval = 100)
    writer = PillowWriter(fps = 3)
    
    ani.save(path +  file_path + "/energy_animation.gif", writer = writer, dpi = 120)
    plt.close(fig)

### View

In [ ]:
# Show evolution of field distribution
Image(path + '/' +  file_path + "/animation.gif")

In [ ]:
# Show evolution of fluctuations
Image(path + '/' +  file_path + "/fluc_animation.gif")

In [ ]:
# Show evolution of energy density
Image(path + '/' +  file_path + "/energy_animation.gif")

## Mode Evolution

### Unit Calculations

In [ ]:
tautab = np.linspace(1, 5, 1000) # Normalized conformal time array, SHOULD START AT 1!
temp = evolution_kinetic_mis(fAGeV, theta1, vheta1, tautab)

In [ ]:
# Axion mass becoming cosmologically relevant
T1MeV = temp['T1MeV']
H1MeV = temp['H1MeV']
R1 = temp['R1']

In [ ]:
# Trapping
TMeV_trap = temp['TMeV_trap']
HMeV_trap = temp['HMeV_trap']
R_trap = temp['R_trap']
mAMeV_trap = temp['mAMeV_trap']
tau_trap = T1MeV / TMeV_trap 

In [ ]:
# New units (see Eröncel et al.)
kappa_star = k_values * R1*H1MeV / (R_trap*mAMeV_trap)
tau_star = 2*HMeV_trap * (((tau**2)/2)/H1MeV)

In [ ]:
tau_star_trap = HMeV_trap * ((tau_trap**2)/H1MeV)

In [ ]:
print(fr'm_*/H_* = {mAMeV_trap/HMeV_trap:.2f}')

In [ ]:
# Internal normalization check based on Parseval theorem

time_idx = 5

real_variance = theta_maps[mask][time_idx].std()**2 # Variance in real space
P_k = 2 * esps[time_idx] / (ma[mask][time_idx] * R[mask][time_idx])**2 # Power spectrum
spec_variance = np.trapezoid(P_k*k_values**2*(1/(2*np.pi**2)), k_values) # Variance in Fourier space

print(f"Ratio of variance in real and Fourier space = {spec_variance / real_variance:.5f}") # Should be close to 1

### Select Time Steps and Momentum Modes

In [ ]:
print(f'Maximal time tau: {tau[mask].max():.2f}')
print(f'Maximal mode momentum kappa_*: {kappa_star.max():.2f}')

In [ ]:
# Choose time steps and find nearest indices
desired_tau_values = [1.0, 2.0, 3.0]
time_indices = [np.argmin((tau[mask] - desired_tau)**2) for desired_tau in desired_tau_values]

# Choose modes and find nearest indices
desired_kappa_star_values = [1, 5, 10]
mode_indices = [np.argmin((kappa_star - desired_kappa_star)**2) for desired_kappa_star in desired_kappa_star_values]

### Spectra

In [ ]:
# Plot dimensionless power spectrum of axion field
fig, ax = plt.subplots(figsize = (10, 6))

for i in time_indices:
    ax.loglog(kappa_star, k_values**3/(2*np.pi**2) * 2*esps[i]/(ma[mask][i]*R[mask][i])**2, label = fr"$\tau_*={tau_star[mask][i]:.2f}$" + "\n" + fr"$\tau={tau[mask][i]:.2f}$")
ax.set_title(
    r'Dimensionless Power Spectrum of Axion Field' + '\n' + 
    r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
    fontsize = 14,
    fontweight = 'bold'
)
ax.set_xlabel(r'$\kappa_\ast$')
ax.set_ylabel(r'$\Delta_{\kappa_{\ast}}^2$')
ax.set_yscale('log')
ax.grid(True, linestyle = ':', alpha = 0.5)
ax.legend(ncols = 3, fontsize = 12, loc = 'lower right')

# Add secondary x-axis
secax = ax.secondary_xaxis('top', functions = (lambda x: np.interp(x, kappa_star, k_values), lambda x: np.interp(x, k_values, kappa_star)))
secax.set_xlabel(r'$k$')

# Avoid overlapping ticks
ticks = secax.get_xticks()
min_spacing = 10**-10 * (ticks[-1] - ticks[0]) # Define minimum spacing threshold
# Filter out ticks that are closer than min_spacing to the previous tick
filtered_ticks = [ticks[0]]
for t in ticks[1:]:
    if t - filtered_ticks[-1] > min_spacing:
        filtered_ticks.append(t)
secax.set_xticks(filtered_ticks)

ax.set_xlim(left = None, right = np.max(kappa_star))

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Plot evolution of modes in trapping units
fig, ax = plt.subplots(figsize = (10, 6))

for i in mode_indices:
    ax.plot(tau_star[mask], (esps[:, i]/(ma[mask]*R[mask])**2) / (esps[0, i]/(ma[mask][0]*R[mask][0])**2), label = fr'$\kappa_\ast = {kappa_star[i]:.2f}$' + '\n' + fr'$k={k_values[i]:.2f}$')

ax.axhline(1, c = 'black', linestyle = '--')
ax.axvline(tau_star_trap, linestyle = '--', c = 'black')
ax.text(tau_star_trap*1.1, plt.ylim()[1]*0.1, r'$TRAPPING$', color = 'black', fontsize = 10)
ax.set_title(
    r'Evolution of Modes' + '\n' + 
    r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
    fontsize = 14,
    fontweight = 'bold'
)
ax.set_xlabel(r'$\tau_\ast$')
ax.set_ylabel(r'$\langle |\theta_{\kappa_{\ast}}|^2 \rangle/\langle |\theta_{\kappa_{\ast},i}|^2 \rangle$')
ax.set_yscale('log')
ax.grid(True, linestyle = ':', alpha = 0.5)
ax.legend(ncols = 3, fontsize = 12, loc = 'lower right')

# Add secondary x-axis
secax = ax.secondary_xaxis('top', functions = (lambda x: np.interp(x, tau_star, tau), lambda x: np.interp(x, tau, tau_star)))
secax.set_xlabel(r'$\tau$')

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Plot dimensionless power spectrum of energy density fluctuations
fig, ax = plt.subplots(figsize = (10, 6))

for i in time_indices:
    ax.loglog(kappa_star, k_values**3 / (np.pi**2) * psps[i], label = fr"$\tau_\ast={tau_star[mask][i]:.2f}$" + "\n" + fr"$\tau={tau[mask][i]:.2f}$")
    
ax.set_title(
    r'Dimensionless Power Spectrum of Density Fluctuations' + '\n' + 
    r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
    fontsize = 14,
    fontweight = 'bold'
)
ax.set_xlabel(r'$\kappa_\ast$')
ax.set_ylabel(r'$\Delta_{\kappa_{\ast}}^2$')
ax.set_yscale('log')
ax.grid(True, linestyle = ':', alpha = 0.5)
ax.legend(ncols = 3, fontsize = 12, loc = 'lower right')

# Add secondary x-axis
secax = ax.secondary_xaxis('top', functions = (lambda x: np.interp(x, kappa_star, k_values), lambda x: np.interp(x, k_values, kappa_star)))
secax.set_xlabel(r'$k$')

# Avoid overlapping ticks
ticks = secax.get_xticks()
min_spacing = 10**-10 * (ticks[-1] - ticks[0]) # Define minimum spacing threshold
# Filter out ticks that are closer than min_spacing to the previous tick
filtered_ticks = [ticks[0]]
for t in ticks[1:]:
    if t - filtered_ticks[-1] > min_spacing:
        filtered_ticks.append(t)
secax.set_xticks(filtered_ticks)
ax.set_xlim(left = None, right = np.max(kappa_star))

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# Plot masked dimensionless power spectrum of energy density fluctuations
fig, ax = plt.subplots(figsize = (10, 6))

for i in time_indices:
    ax.loglog(kappa_star, k_values**3 / (np.pi**2) * pspVs[i], label = fr"$\tau_\ast={tau_star[mask][i]:.2f}$" + "\n" + fr"$\tau={tau[mask][i]:.2f}$")
    
ax.set_title(
    r'Masked (AXITV) Dimensionless Power Spectrum of Density Fluctuations' + '\n' + 
    r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
    fontsize = 14,
    fontweight = 'bold'
)
ax.set_xlabel(r'$\kappa_\ast$')
ax.set_ylabel(r'$\Delta_{\kappa_{\ast}}^2$')
ax.set_yscale('log')
ax.grid(True, linestyle = ':', alpha = 0.5)
ax.legend(ncols = 3, fontsize = 12, loc = 'lower right')

# Add secondary x-axis
secax = ax.secondary_xaxis('top', functions = (lambda x: np.interp(x, kappa_star, k_values), lambda x: np.interp(x, k_values, kappa_star)))
secax.set_xlabel(r'$k$')

# Avoid overlapping ticks
ticks = secax.get_xticks()
min_spacing = 10**-10 * (ticks[-1] - ticks[0]) # Define minimum spacing threshold
# Filter out ticks that are closer than min_spacing to the previous tick
filtered_ticks = [ticks[0]]
for t in ticks[1:]:
    if t - filtered_ticks[-1] > min_spacing:
        filtered_ticks.append(t)
secax.set_xticks(filtered_ticks)
ax.set_xlim(left = None, right = np.max(kappa_star))

plt.tight_layout()
plt.show()
plt.close()

## Axitons

In [ ]:
# Plot energy density distribution
plt.figure(figsize = (6, 6))
plt.imshow(theta_proj_maps[-1], cmap = pa.thetacmap, norm = LogNorm())
plt.colorbar(label = r'$\sum_z \rho$')
plt.xlabel(r'$x$')  
plt.ylabel(r'$y$')  
plt.title(r'Energy Density Projection Distribution' + '\n' + 
          r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
          fontsize = 12, fontweight = 'bold')
plt.show()
plt.close()

In [ ]:
# Plot axitons
mf_axi = mf[-1]
axitons = find_all_axitons(mf_axi, 10, 0.99, flat = False)
p = pa.gm(mf_axi, '2DmapP')

for axi_idx in axitons:
    plt.figure(figsize = (6, 6))
    size = 64
    y, x = axi_idx # Coordinate indices of axiton
    # Zoomed-in and centered around axiton
    plt.imshow(p[y-size:y+size, x-size:x+size], cmap = pa.thetacmap, origin = 'lower', 
               extent = [x-size, x+size, y-size, y+size], norm = LogNorm())
    #plt.colorbar(label = r'$\sum_z \rho$')
    plt.xlabel(r'$x$')  
    plt.ylabel(r'$y$')  
    plt.title(r'Axiton' + '\n' + 
              r'$f_a = {:.0e}$ GeV, $\Theta_1 = {:.2f}$, $\dot{{\Theta}}_1/H_1 = {:.2f}$'.format(fAGeV, theta1, vheta1),
              fontsize = 12, fontweight = 'bold')
    plt.show()